# 喜马拉雅有声书 Colab Worker

轮询 VPS 认领任务并处理：下载音频 → DeepFilter 降噪 → 上传 Telegram → 上报结果。

## 工作流程
1. 轮询 VPS API 认领任务 (`GET /api/jobs/claim`)
2. 对每个章节：
   - 下载音频 (喜马拉雅 mobile-playpage API + AES 解密)
   - DeepFilter 降噪 (可选)
   - 上传到 Telegram (Bot API `sendAudio`)
   - 上报结果 (`POST /api/jobs/{job_id}/chapter`)
3. 全部完成后标记任务完成 (`POST /api/jobs/{job_id}/complete`)

> **使用方法**：从上到下依次执行每个 Cell。

## 1. 安装依赖

In [ ]:
!pip install -q requests pycryptodome tqdm pydub

## 2. 配置参数

修改下方参数后执行：

In [ ]:
# ═══ VPS 连接配置 ═══
VPS_URL = "http://your-vps:59388"
WORKER_ID = "colab_001"        # 留空则自动生成
WORKER_TOKEN = "your_worker_token"

# ═══ 运行参数 ═══
POLL_INTERVAL = 10              # 无任务时等待秒数
MAX_JOBS = 0                    # 最大处理任务数 (0=不限)

# ═══ 代码来源 (二选一) ═══
# 方式 A: 从 GitHub 克隆 (推荐)
GIT_REPO = ""                   # 如 https://github.com/user/ximalaya_manager.git

# 方式 B: 使用已上传到 /content 的代码 (留空 GIT_REPO 则使用此方式)
# 确保pipeline目录在 /content/ximalaya_manager/pipeline

## 3. 获取代码 & 设置路径

In [ ]:
import os, sys

if GIT_REPO:
    !rm -rf /content/ximalaya_manager
    !git clone {GIT_REPO} /content/ximalaya_manager

# 确保 pipeline 包可导入
for candidate in ["/content/ximalaya_manager", "/content", "/app"]:
    if os.path.isdir(os.path.join(candidate, "pipeline")):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        print(f"pipeline 路径: {candidate}")
        break
else:
    print("警告: 未找到 pipeline 目录，请确认代码已上传")

## 4. 导入模块 & 日志配置

In [ ]:
from __future__ import annotations

import os, sys, time, json, shutil, tempfile
import logging, threading, requests

# 修复 Colab 控制台编码
if hasattr(sys, 'stdout'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    sys.stderr.reconfigure(encoding='utf-8', errors='replace')

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("colab_worker")
print("模块导入完成")

## 5. ColabWorker 类定义

In [ ]:
class ColabWorker:
    """Colab Worker 客户端。"""

    def __init__(self, vps_url: str, worker_id: str, worker_token: str):
        self.vps_url = vps_url.rstrip("/")
        self.worker_id = worker_id
        self.worker_token = worker_token
        self.config: dict = {}
        self._heartbeat_stop = threading.Event()

    # ─── HTTP 工具 ───

    def _get(self, path: str, params: dict | None = None) -> dict:
        params = params or {}
        params["worker_token"] = self.worker_token
        url = f"{self.vps_url}{path}"
        resp = requests.get(url, params=params, timeout=30)
        return resp.json()

    def _post(self, path: str, data: dict | None = None) -> dict:
        url = f"{self.vps_url}{path}"
        resp = requests.post(
            url,
            json=data or {},
            params={"worker_token": self.worker_token},
            timeout=30,
        )
        return resp.json()

    # ─── 心跳 ───

    def _heartbeat_loop(self):
        while not self._heartbeat_stop.is_set():
            try:
                self._get("/api/worker/heartbeat", {"worker_id": self.worker_id})
            except Exception:
                pass
            self._heartbeat_stop.wait(30)

    def start_heartbeat(self):
        t = threading.Thread(target=self._heartbeat_loop, daemon=True)
        t.start()
        logger.info("心跳线程已启动")

    def stop_heartbeat(self):
        self._heartbeat_stop.set()

    # ─── 配置 ───

    def fetch_config(self) -> dict:
        try:
            resp = self._get("/api/config", {"worker_id": self.worker_id})
            if resp.get("ok"):
                self.config = resp.get("config", {})
                logger.info(f"配置获取成功: TG tokens={len(self.config.get('tg_bot_tokens', []))}, "
                           f"deepfilter={self.config.get('enable_deepfilter', True)}")
                return self.config
        except Exception as e:
            logger.error(f"配置获取失败: {e}")
        return {}

    # ─── 任务认领 ───

    def claim_job(self) -> dict | None:
        try:
            resp = self._get("/api/jobs/claim", {"worker_id": self.worker_id})
            if resp.get("ok"):
                job = resp.get("job", {})
                logger.info(f"认领任务 #{job.get('job_id')}: {job.get('book_name', '')}")
                return job
        except Exception as e:
            logger.error(f"认领任务失败: {e}")
        return None

    # ─── 章节处理 ───

    def process_chapter(self, job_id: int, chapter: dict, book_id: str) -> dict:
        chapter_id = chapter["chapter_id"]
        chapter_name = chapter.get("chapter_name", "")
        track_id = chapter_id

        logger.info(f"  下载章节: {chapter_name} (trackId={track_id})")

        tmp_dir = tempfile.mkdtemp(prefix="xm_chapter_")
        audio_path = os.path.join(tmp_dir, f"{chapter.get('chapter_order', 0):04d}_{track_id}.m4a")

        try:
            from pipeline.ximalaya_api import download_track

            cookie = self.config.get("xm_cookie", "")
            headers = {"Cookie": cookie} if cookie else None
            download_interval = self.config.get("download_interval", 1.5)

            status, file_size = download_track(track_id, audio_path, headers=headers)
            if status not in ("downloaded", "skipped"):
                return self._report_chapter(job_id, chapter_id, "failed", error_message=f"下载失败: {status}")
            if status == "skipped" and file_size < 1000:
                return self._report_chapter(job_id, chapter_id, "failed", error_message="文件太小")

            time.sleep(download_interval)

            # ─── DeepFilter 降噪 ───
            if self.config.get("enable_deepfilter", True):
                logger.info(f"  降噪中: {chapter_name}")
                try:
                    from pipeline.deepfilter import denoise_audio_keep_format, setup_deep_filter

                    if not os.path.exists(
                        os.path.join(os.environ.get("DEEPFILTER_DIR", "/content/.deepfilter"),
                                     "deep-filter-0.5.6-x86_64-unknown-linux-musl")
                    ):
                        setup_deep_filter()

                    seg_min = self.config.get("deepfilter_segment_minutes", 60)
                    denoised_path = audio_path.replace(".m4a", "_denoised.m4a")
                    denoised_path = denoise_audio_keep_format(audio_path, denoised_path, seg_min)

                    if os.path.exists(denoised_path) and os.path.getsize(denoised_path) > 0:
                        audio_path = denoised_path
                except Exception as e:
                    logger.warning(f"  降噪失败，使用原始音频: {e}")

            # ─── 上传到 Telegram ───
            logger.info(f"  上传TG: {chapter_name}")
            from pipeline.tg_upload import upload_with_token_rotation

            bot_tokens = self.config.get("tg_bot_tokens", [])
            chat_id = self.config.get("tg_chat_id", "")
            serial = self.config.get("tg_serial_upload", True)
            interval = self.config.get("tg_upload_interval", 3.0)

            if not bot_tokens or not chat_id:
                return self._report_chapter(job_id, chapter_id, "failed",
                                           error_message="TG Bot Token 或 Chat ID 未配置")

            result = upload_with_token_rotation(
                file_path=audio_path,
                bot_tokens=bot_tokens,
                chat_id=chat_id,
                title=chapter_name[:64],
                caption=chapter_name,
                serial=serial,
                interval=interval,
            )

            if not result.get("ok"):
                return self._report_chapter(job_id, chapter_id, "failed",
                                           error_message=result.get("error", "上传失败"))

            return self._report_chapter(job_id, chapter_id, "uploaded",
                                       telegram_file_id=result.get("file_id", ""),
                                       telegram_message_id=result.get("message_id", 0),
                                       telegram_bot_id=None,
                                       telegram_bot_user_id=result.get("bot_user_id"))

        except Exception as e:
            logger.error(f"  章节处理异常: {e}", exc_info=True)
            return self._report_chapter(job_id, chapter_id, "failed", error_message=str(e))
        finally:
            shutil.rmtree(tmp_dir, ignore_errors=True)

    def _report_chapter(self, job_id: int, chapter_id: str, upload_status: str,
                        telegram_file_id: str = "", telegram_message_id: int = 0,
                        telegram_bot_id: int | None = None,
                        telegram_bot_user_id: int | None = None,
                        error_message: str = "") -> dict:
        try:
            resp = self._post(f"/api/jobs/{job_id}/chapter", {
                "chapter_id": str(chapter_id),
                "upload_status": upload_status,
                "telegram_file_id": telegram_file_id,
                "telegram_message_id": telegram_message_id,
                "telegram_bot_id": telegram_bot_id,
                "telegram_bot_user_id": telegram_bot_user_id,
                "error_message": error_message,
            })
            status_text = "OK" if upload_status == "uploaded" else "FAIL"
            logger.info(f"  [{status_text}] 章节 {chapter_id}: {upload_status}"
                       + (f" file_id={telegram_file_id[:20]}..." if telegram_file_id else "")
                       + (f" err={error_message}" if error_message else ""))
            return resp
        except Exception as e:
            logger.error(f"  上报失败: {e}")
            return {"ok": False, "error": str(e)}

    # ─── 任务完成 ───

    def complete_job(self, job_id: int, result: dict | None = None):
        resp = self._post(f"/api/jobs/{job_id}/complete", {"result": result})
        logger.info(f"任务 #{job_id} 已完成")
        return resp

    def fail_job(self, job_id: int, error_message: str):
        resp = self._post(f"/api/jobs/{job_id}/fail", {"error_message": error_message})
        logger.error(f"任务 #{job_id} 失败: {error_message}")
        return resp

    # ─── 主循环 ───

    def run(self, poll_interval: int = 10, max_jobs: int = 0):
        logger.info(f"Colab Worker 启动: {self.worker_id}")
        logger.info(f"VPS: {self.vps_url}")

        self.fetch_config()
        self.start_heartbeat()

        jobs_done = 0
        try:
            while True:
                if max_jobs > 0 and jobs_done >= max_jobs:
                    logger.info(f"已处理 {jobs_done} 个任务，退出")
                    break

                job = self.claim_job()
                if not job:
                    logger.info(f"无待处理任务，等待 {poll_interval}s...")
                    time.sleep(poll_interval)
                    continue

                job_id = job["job_id"]
                book_id = job.get("book_id", "")
                chapters = job.get("chapters", [])
                total = len(chapters)

                if not chapters:
                    self.complete_job(job_id, {"note": "no pending chapters"})
                    jobs_done += 1
                    continue

                logger.info(f"开始处理任务 #{job_id}: {job.get('book_name', '')} ({total} 章节)")

                self.fetch_config()

                success_count = 0
                fail_count = 0

                for i, chapter in enumerate(chapters):
                    logger.info(f"  [{i+1}/{total}] {chapter.get('chapter_name', '')}")
                    result = self.process_chapter(job_id, chapter, book_id)

                    if chapter.get("upload_status") == "uploaded" or (result and result.get("upload_status") == "uploaded"):
                        success_count += 1
                    else:
                        fail_count += 1

                if fail_count == 0:
                    self.complete_job(job_id, {"success": success_count, "failed": fail_count})
                else:
                    self.complete_job(job_id, {"success": success_count, "failed": fail_count,
                                               "note": f"{fail_count} chapters failed"})

                jobs_done += 1
                logger.info(f"任务 #{job_id} 完成: 成功={success_count}, 失败={fail_count}")

        except KeyboardInterrupt:
            logger.info("用户中断")
        finally:
            self.stop_heartbeat()
            logger.info("Worker 已停止")

print("ColabWorker 类定义完成")

## 6. 启动 Worker

执行后开始轮询任务。`MAX_JOBS=0` 表示无限循环，设为具体数字则处理完后自动停止。

In [ ]:
# 生成 Worker ID
worker_id = WORKER_ID or f"colab_{os.urandom(4).hex()}"

worker = ColabWorker(
    vps_url=VPS_URL,
    worker_id=worker_id,
    worker_token=WORKER_TOKEN,
)

worker.run(poll_interval=POLL_INTERVAL, max_jobs=MAX_JOBS)

## 7. 手动停止 Worker (可选)

如果 Worker 在后台运行或需要手动停止，执行此 Cell。

In [ ]:
worker.stop_heartbeat()
print("Worker 心跳已停止")